# ALIGN-PARETO — M2: DPO alignment of Qwen3-0.6B
Trains a LoRA adapter on **Anthropic/hh-rlhf** preference pairs with TRL's `DPOTrainer` (4-bit QLoRA base,
spec §18–19). The adapter is the full alignment update; M3 scales it with λ via `src/alignment/adapters.py`.

**Kaggle setup:** input `align-pareto-src` (code + `data/eval/eval_prompts.csv`) attached, **GPU T4**, internet on
(Qwen3-0.6B and hh-rlhf are downloaded from the Hugging Face Hub; both are public). Run from this machine with
`kaggle/run_on_kaggle.sh m2`. All logic lives in `src/`; this notebook only runs it and shows the results.

In [ ]:
# 1. Environment (one GPU: DataParallel over two T4s does not work with 4-bit weights)
import os
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")
import sys, json, platform, shutil, glob, subprocess
from pathlib import Path
import importlib.metadata as md

print("Python", platform.python_version())
import torch
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0), f"| VRAM {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
else:
    print("No GPU: M2 needs one (Settings -> Accelerator -> GPU T4)")

In [ ]:
# 2. Dependencies (pinned to the versions the code was tested with)
PINNED = ["transformers==5.22.2", "trl==1.17.1", "peft==0.21.2", "bitsandbytes", "datasets"]
subprocess.run([sys.executable, "-m", "pip", "install", "-q", *PINNED], check=True)
for pkg in ["torch", "transformers", "trl", "peft", "bitsandbytes", "accelerate", "datasets", "huggingface_hub"]:
    try:
        print(f"{pkg:16s} {md.version(pkg)}")
    except md.PackageNotFoundError:
        print(f"{pkg:16s} MISSING")

In [ ]:
# 3. Project code: Kaggle dataset `align-pareto-src` -> /kaggle/working/align-pareto; locally, the repo itself
ON_KAGGLE = Path("/kaggle/working").exists()
if ON_KAGGLE:
    hits = glob.glob("/kaggle/input/**/src/run_m2.py", recursive=True)
    assert hits, "Add the dataset 'align-pareto-src' to this notebook (Add Input)."
    ROOT = Path("/kaggle/working/align-pareto")
    shutil.copytree(Path(hits[0]).parents[1], ROOT, dirs_exist_ok=True)
else:
    ROOT = Path.cwd().parent
assert (ROOT / "data/eval/eval_prompts.csv").exists(), "data/eval/eval_prompts.csv missing: re-upload align-pareto-src"
sys.path.insert(0, str(ROOT))
print("Project root:", ROOT)

In [ ]:
# 4. Run M2 (set QUICK=True for a 3-step smoke test)
QUICK = False
from src.run_m2 import run_m2
metrics = run_m2(ROOT, quick=QUICK)
RES = ROOT / "results" / ("m2_quick" if QUICK else "m2")
if ON_KAGGLE:  # keep kernel output small: raw downloads and optimizer checkpoints are not needed afterwards
    shutil.rmtree(ROOT / "data/raw/hf_cache", ignore_errors=True)
    shutil.rmtree(ROOT / "models" / ("m2_quick" if QUICK else "m2") / "checkpoints", ignore_errors=True)

In [ ]:
# 5. Training checks: loss should fall, reward accuracy rise above 0.5, the adapter reload match
print(json.dumps({k: metrics[k] for k in ["train_steps", "train_minutes", "train_loss", "train_reward_accuracy",
                                          "train_reward_margin", "final_eval", "reload_check"]}, indent=2))
from IPython.display import Image, display
display(Image(filename=str(RES / "plots/dpo_training.png"), width=900))

In [ ]:
# 6. Smoke test: greedy replies, base (lambda=0) vs aligned (lambda=1), two validation prompts per quadrant
import pandas as pd
pd.set_option("display.max_colwidth", 300)
print("Share of prompts where the replies differ:", metrics["smoke_test"])
display(pd.read_csv(RES / "smoke_test.csv")[["quadrant", "prompt", "reply_lambda_0", "reply_lambda_1"]])

## Next: M3
`models/m2/dpo_adapter` is loaded by the λ sweep (`notebooks/03_lambda_sweep_kaggle.ipynb`) with
`train_dpo.load_adapter_model` and scaled with `adapters.set_peft_lora_lambda`.